


### Search Engine With Tools And Agents

In [ ]:

from dotenv import load_dotenv
import os

from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# from langchain.tools.retriever import create_retriever_tool
from langchain.agents import create_react_agent, AgentExecutor
# from langchain_community.tools.retriever import create_retriever_tool
from langchain.tools import Tool
from langchain_groq import ChatGroq

from langchain.agents import initialize_agent
from langchain.agents import AgentType
from langchain import hub






ImportError: cannot import name 'create_react_agent' from 'langchain.agents' (c:\Users\ADMIN\miniconda3\envs\ann_env\lib\site-packages\langchain\agents\__init__.py)

In [ ]:

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
print("API KEY Loaded:", groq_api_key is not None)

API KEY Loaded: True


In [ ]:

# Create a Wikipedia API wrapper that controls how results are fetched.

wiki_wrapper = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=300
)


wiki_tool = WikipediaQueryRun(
    api_wrapper=wiki_wrapper # api_wrapper inbuilt with configuration for how to fetch results needed to create 
)

wiki_tool.name

'wikipedia'

In [ ]:

arxiv_wrapper = ArxivAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=300
)

arxiv_tool = ArxivQueryRun(
    api_wrapper=arxiv_wrapper
)

arxiv_tool.name

'arxiv'

In [ ]:

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    documents,
    embeddings
)
retriever = vectorstore.as_retriever()


retriever_tool = create_retriever_tool(
    retriever,
    name="langsmith-search",
    description="Search any information about LangSmith"
)

retriever_tool.name

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'langsmith-search'

In [ ]:

tools = [
    wiki_tool,
    arxiv_tool,
    retriever_tool
]

tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\ADMIN\\miniconda3\\envs\\ann_env\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=300)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=300)),
 Tool(name='langsmith-search', description='Search any information about LangSmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x00000175916A3E20>, retriever=VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000175

In [ ]:

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0
)

llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000175C05A2350>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000175C0645870>, model_name='llama-3.1-8b-instant', temperature=1e-08, groq_api_key=SecretStr('**********'))

In [ ]:
# load a pre-built prompt template from the LangChain Hub instead of writing the prompt manually.
prompt = hub.pull("hwchase17/react")
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [ ]:

agent = create_react_agent( 
    llm=llm,
    tools=tools,
    prompt=prompt
)

agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], partial_variables={'tools': "wikipedia - A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.\narxiv - A wrapper around Arxiv.org Useful for when you need to answer questions about Physics, Mathematics, Computer Science, Quantitative Biology, Quantitative Finance, Statistics, Electrical Engineering, and Economics from scientific articles on arxiv.org. Input should be a search query.\nlangsmith-search(query: 'str', *, retriever: 'BaseRetriever' = VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000175C02875E0>), document_prompt: 'BasePromptTemplate' = PromptTemplate(input_variables=['page_co

In [ ]:

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

agent_executor

AgentExecutor(verbose=True, agent=RunnableAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], partial_variables={'tools': "wikipedia - A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.\narxiv - A wrapper around Arxiv.org Useful for when you need to answer questions about Physics, Mathematics, Computer Science, Quantitative Biology, Quantitative Finance, Statistics, Electrical Engineering, and Economics from scientific articles on arxiv.org. Input should be a search query.\nlangsmith-search(query: 'str', *, retriever: 'BaseRetriever' = VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000175C02875E0>), document_prompt: 'BaseP

In [ ]:

response = agent_executor.invoke({
    "input": "Tell me about LangSmith"
})

print("\nFINAL ANSWER:\n")
print(response["output"])



> Entering new AgentExecutor chain...
Thought: I need to find information about LangSmith, which seems to be a tool or a platform. I should check if it's a general topic or a specific one related to a particular field.

Action: langsmith-search
Action Input: 'LangSmith'LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantBetaCLISkillsAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.
It helps you trace requests, evaluate outputs, test pro

In [ ]:
# Test more queries

agent_executor.invoke({"input": "What is Machine Learning?"})




> Entering new AgentExecutor chain...
Thought: I need to find a definition of Machine Learning. Wikipedia seems like a good place to start.

Action: wikipedia
Action Input: Machine LearningPage: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit instructions. Within a subdisciplineQuestion: What is Machine Learning?
Thought: I need to find a definition of Machine Learning. Wikipedia seems like a good place to start.

Action: wikipedia
Action Input: Machine LearningPage: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit instructions. Within a subdisciplineQuestion: What is Mach

{'input': 'What is paper 1706.03762 about?',
 'output': 'Agent stopped due to iteration limit or time limit.'}

In [ ]:
agent_executor.invoke({"input": "What is paper 1706.03762 about?"})